In [ ]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")
from mid_structure_square import create_mid_structure_square 

import gdsfactory as gf
cell_temp = gf.Component()
# the whole chip
block = gf.Component()

2026-05-27 17:58:54.397 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/3962406160.oas'


In [2]:
# square mesh
L = [1000]
b = [500]
g = [48]
w = 2


mesh_rect = gf.Component()
origin = (0, 0)

for i in range(len(L)):
    for j in range(len(b)):
        column = int(b[j] / (w + g[i]))
        row = int(L[j] / (w + g[i]))
        for n in range(row):
            r = gf.components.rectangle(size=(b[i]+w, w), layer=(9, 0))
            (mesh_rect << r).movey((n+1) * (w + g[i])-w)
        for m in range(column+1):
            r = gf.components.rectangle(size=(w, L[i]), layer=(9, 0))
            (mesh_rect << r).movex(m * (w + g[i]))

    # length mark
    T = gf.components.text(f"L={L[i]} b={b[i]} gap={g[i]}", size=50, layer=(1, 0))
    T_ref = block << T
    T_ref.move((-350, -1500))




In [4]:
# note
"""
layer 9: structure to keep
layer 11: gold deposition area
layer 12: frontside etching area
"""

'\nlayer 9: structure to keep\nlayer 11: gold deposition area\nlayer 12: frontside etching area\n'

In [3]:
# web mesh
mesh_web = gf.Component()
sections = []

n_horizontal = 18
for i in range(n_horizontal):
    offset = i * 50
    s = gf.Section(width=2, offset=offset, layer=(9, 0))
    sections.append(s)

x_circle = gf.CrossSection(sections=tuple(sections))

p_circle = gf.path.arc(50)
# Combine the Path and the cross-section.
b_circle = gf.path.extrude(p_circle, cross_section=x_circle)    
(mesh_web << b_circle).move((0, -50))


support = gf.components.rectangle(size=(2, (g[0]+w)*(n_horizontal-1)), layer=(9, 0))
for i in range(2):
    angle = 90 / 3 * (i+1) + 180
    (mesh_web << support).move((0, 50)).rotate(angle=angle, center=(0, 0))
# mesh_web.show()




In [4]:
# web combination
web = gf.Component()
web_temp = gf.Component()
(web_temp << mesh_rect).move((-(b[0]+w)/2, -L[0]-300))
(web_temp << mesh_web).move(((b[0]+w)/2, -301+50))
for i in range(4):
    angle = 90 * i
    (web << web_temp).rotate(angle=angle, center=(0, 0))
block << web




Unnamed_1: ports [], vinsts=[] info=Info() kcl=KCLayout(name='DEFAULT', layout=<klayout.dbcore.Layout object at 0x1170a07d0>, layer_enclosures=LayerEnclosureModel(root={'4fb2b992': LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure')}), cross_sections={'4fb2b992_2000': SymmetricalCrossSection(width=2000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_2000'), '4fb2b992_502000': SymmetricalCrossSection(width=502000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_502000'), '4fb2b992_1000000': SymmetricalCrossSection(width=1000000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_1000000'), '4fb2b992_850000': SymmetricalCrossSection(width=850000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_850000')}, enclosure=KCellEnclosure(enclosures=LayerEnclosureCollection(encl

In [ ]:
# mid structure
mid_struct = create_mid_structure_square()
# mid_struct.show()
block << mid_struct

Unnamed_1: ports [], vinsts=[] info=Info() kcl=KCLayout(name='DEFAULT', layout=<klayout.dbcore.Layout object at 0x1170a07d0>, layer_enclosures=LayerEnclosureModel(root={'4fb2b992': LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), '76156b2a': LayerEnclosure(layer_sections={}, main_layer=11/0, yaml_tag='!Enclosure')}), cross_sections={'4fb2b992_2000': SymmetricalCrossSection(width=2000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_2000'), '4fb2b992_502000': SymmetricalCrossSection(width=502000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_502000'), '4fb2b992_1000000': SymmetricalCrossSection(width=1000000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_1000000'), '4fb2b992_850000': SymmetricalCrossSection(width=850000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name

In [ ]:
# frontside etching area
frame_size = 2600
frontside = gf.components.rectangle(size=(frame_size, frame_size), layer=(12, 0))
(block << frontside).move((-1300, -1300))

# backside etching area
backside_size = frame_size + 743.44
backside = gf.components.rectangle(size=(backside_size, backside_size), layer=(3, 0))
(block << backside).move((-backside_size/2, -backside_size/2))

# side frame etching area
# 7300um x 7300um gap between frame: 600
etch_frame = gf.components.rectangle(size=(6100, 375), layer=(3, 0))
for i in range(4):
    angle = 90 * i
    (block << etch_frame).move((-6100/2, -7300/2)).rotate(angle=angle, center=(0, 0))
# block.show()

In [ ]:
def create_simulation_structure():
    simu_structure = gf.boolean(A = block, B = block,  operation="not", layer1=(9, 0), layer2=(100, 0), layer=(1, 0))
    return simu_structure

In [9]:
# order
def add_order_text(order_number):
    order = gf.Component()
    for j in range(4):
        T = gf.components.text(f"MDL{order_number}", size=20, layer=(1, 0))
        order_ref = order << T
        if j == 0:
            order_ref.move((-2000, -2000))
        elif j == 1:
            order_ref.move((-2000, 2000))
        elif j == 2:
            order_ref.move((2000, -2000))
        else:
            order_ref.move((2000, 2000))
    return order
    # order.show()
        

In [10]:
# repeat
fblock = gf.Component()
block_temp = gf.Component()
(block_temp << block)
for i in range(1):
    if i == 0:
        block_ref = fblock << block_temp
    elif i == 1:
        block_ref = fblock << block_temp
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block_temp
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block_temp
        block_ref.move((0, 10000))

In [ ]:
# boolean operation
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(12, 0), layer2=(9, 0), layer=(1, 0))
cell_temp << outside

#  gold
gold = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(11, 0), layer2=(30, 0), layer=(2, 0))
cell_temp << gold
# backside etching
backside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(30, 0), layer=(3, 0))
cell_temp << backside
# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(30, 0), layer=(1, 0))
cell_temp << marker
# frame
# frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
# frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
# frame2_ref = cell_temp << frame2
# frame2_ref.move((-2500, -2500))
# cell_temp << frame1

def cell_mesh_design_less(order_number):
    cell_mesh_design_less = gf.Component()
    cell_temp_temp = gf.Component()
    order = add_order_text(order_number)
    cell_temp_temp << order
    cell_temp_temp << cell_temp
    cell_mesh_design_less << cell_temp_temp
    return cell_mesh_design_less
    # cell_ref.move((2500, -7500))
    # cell_mesh_design_less.show()
    # cell_mesh_design_less.write_gds("mesh.gds")
    # cell_mesh_design_less.plot()

if __name__ == "__main__":
    cell = cell_mesh_design_less(order_number=1)
    cell.show()